# 02 — デモ: Session 2 平坦パラメータチューニング

**目的:** MPC設計者として **μ / step_freq / gain** を触り、成功と失敗の体感を得る。


## 4セッションの違い（必読）

| | **S1 本Notebook** | **S2 tune ← 今ここ** | S3a boxes | S3b perlin |
|---|-------------------|---------|-----------|------------|
| **scene** | **flat（平坦）** | flat | **random_boxes（箱）** | **perlin（連続起伏）** |
| **足場最適化** | **OFF** | OFF | ON | ON |
| **主な目的** | 最小構成で動作確認 | μ / 歩調チューニング | 段差・離散障害 | 連続起伏 |
| **デモGIFで見る点** | 平坦＋標準trot | 平坦＋**速いtrot** | **箱が見える** | **うねり地形** |
| **GIF** | demo_s01_flat | demo_s02_tune | demo_s03_boxes | demo_s03_perlin |

> **S1 と S2 は地形とも平坦**です。GIFの違いは **歩調（S2は step_freq=1.75 Hz の速い trot）** と **Notebook内の実験内容** です。  
> **S3a/S3b は約9秒走行**して箱・起伏地形に入るようキャプチャしています（旧GIFは短すぎて全部平坦に見えていました）。


### このセッション固有のポイント

- **地形:** S1 と同じ `scene=flat`（平坦）。GIF だけでは S1 と区別しにくい  
- **GIFとの違い:** step_freq=**1.75 Hz**（S1 の 1.4 Hz より速い trot → 足振りが速く見える）  
- **Notebookとの違い:** μ / step_freq / duty_factor を **数値スイープ** して成功・失敗パターンを体感  
- 画面左上: `Session 2 | scene=flat | step_freq=1.75 Hz (fast trot)`

![Session 2 demo](../assets/demo_s02_tune.gif)


## Step 0 — このデモのゴール

| 触るパラメータ | 体感すべきこと |
|----------------|----------------|
| `mu` | 加速 vs 転倒のトレードオフ |
| `step_freq` | 歩調と MPC 追従性 |
| `duty_factor` | 安定 vs 敏捷 |


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")


## Step 1 — ベースラインプリセット適用

In [ ]:
apply_preset("session02_flat_tune")
preset = load_preset_yaml("session02_flat_tune")
print("tuning hints:", preset.get("tuning_hints"))


## Step 2 — ベースライン計測（4秒）

In [ ]:
baseline = run_flat_sim(seconds=4.0)
print(baseline)
fig = compare_runs([("baseline", baseline)])
plt.show()


## Step 3 — ❌ vs ✅ 実験 A: 摩擦係数 mu

**仮説:** μ↑ → 水平GRFを取りやすく加速するが、高すぎると不安定

| ケース | mu | 期待 |
|--------|-----|------|
| 保守的 | 0.35 | 安定、加速弱い |
| 標準 | 0.5 | バランス |
| 積極 | 0.65 | 加速↑、転倒リスク↑ |


In [ ]:
apply_preset("session02_flat_tune")
runs = []
for mu, label in [(0.35, "mu=0.35 conservative"), (0.5, "mu=0.5 baseline"), (0.65, "mu=0.65 aggressive")]:
    m = run_flat_sim(seconds=4.0, mu=mu)
    runs.append((label, m))
    print(label, "mean_vx=", f"{m['mean_vx']:.3f}", "terminated=", m["terminated"])

fig = compare_runs(runs)
plt.suptitle("mu sweep — vx vs stability (check terminated flag)", y=1.02)
plt.show()


**MPC設計者メモ — mu**

- ✅ 成功: 狙い速度に近い、姿勢安定、`terminated=False`  
- ❌ 失敗: 横滑り・ピッチ/ロール増大 → **μを下げる**  
- ❌ 失敗: 加速不足 → μ↑を試すが、**sim地面摩擦も確認**


## Step 4 — ❌ vs ✅ 実験 B: 歩調 step_freq

**仮説:** 速すぎると MPC の 0.24s 先読みが追いつかない


In [ ]:
apply_preset("session02_flat_tune")
runs = []
for freq, label in [(1.0, "1.0Hz slow"), (1.4, "1.4Hz baseline"), (1.8, "1.8Hz fast")]:
    m = run_flat_sim(seconds=4.0, step_freq=freq)
    runs.append((label, m))
    print(label, "mean_vx=", f"{m['mean_vx']:.3f}", "max_roll=", f"{m['max_roll_deg']:.1f}°")

fig = compare_runs(runs)
plt.show()


**MPC設計者メモ — step_freq**

- ✅ 成功: 周期安定、roll 小さい  
- ❌ 失敗: 足刺さり・振動 → **step_freq↓** or **duty_factor↑**  
- ❌ 失敗: MPC solve timeout → `solver_mode='speed'`


## Step 5 — パラメータスタディ結果（事前計測データ）

In [ ]:
results = load_param_study()
df = pd.DataFrame(results)
display(df)

mu_df = df[df["step_freq"] == 1.4].sort_values("mu")
freq_df = df[df["mu"] == 0.5].sort_values("step_freq")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mu_df["mu"], mu_df["mean_vx"], "o-")
axes[0].set(xlabel="mu", ylabel="mean vx [m/s]", title="mu sweep (6s headless)")
axes[1].plot(freq_df["step_freq"], freq_df["mean_vx"], "s-", color="green")
axes[1].set(xlabel="step_freq [Hz]", ylabel="mean vx [m/s]", title="step_freq sweep")
plt.tight_layout()


> 6秒 headless ではトレンドが単調にならないことがある。**Step 3–4 の比較実験** の方が体感に適す。

## Step 6 — ❌ vs ✅ 実験 C: duty_factor（支持比率）

不整地前に平坦で体感しておくと Session 3 が楽。


In [ ]:
apply_preset("session02_flat_tune")
runs = []
for duty, label in [(0.55, "duty=0.55 short stance"), (0.65, "duty=0.65 baseline"), (0.75, "duty=0.75 long stance")]:
    m = run_flat_sim(seconds=4.0, duty_factor=duty)
    runs.append((label, m))
fig = compare_runs(runs)
plt.show()


## Step 7 — 自分で1つ設計（成功体験）

**課題:** 「安定優先で vx=0.3m/s 程度」を目指すパラメータを1組決めよ。

ヒント: `mu=0.45`, `step_freq=1.2`, `duty_factor=0.72`


In [ ]:
# ここを編集して試す
MY_MU = 0.45
MY_FREQ = 1.2
MY_DUTY = 0.72

apply_preset("session02_flat_tune")
mine = run_flat_sim(seconds=5.0, mu=MY_MU, step_freq=MY_FREQ, duty_factor=MY_DUTY)
print("My config:", mine)
fig = compare_runs([("baseline", baseline), ("my design", mine)])
plt.show()


## Step 8 — チェックリスト

- [ ] μ を1回変えて効果を説明できる  
- [ ] step_freq を1回変えて効果を説明できる  
- [ ] 転倒時に TUNING_GUIDE から **最初の対処** を選べる  
- [ ] S1 との違い（GIF=速いtrot、Notebook=パラメータ実験）を説明できる  

**次:** [03_demo_session03a_rough_boxes.ipynb](./03_demo_session03a_rough_boxes.ipynb)


In [ ]:
pd.DataFrame(TUNING_GUIDE)[["param", "failure_symptom", "failure_fix"]]
